# Ridge Regression Volatility Backtest

CPU-only walk-forward backtest with rolling robust scaling and HAR features.

In [ ]:
import os
import subprocess
import sys

# Clone repo if not present
if not os.path.exists("harxhar"):
    subprocess.run(["git", "clone", "https://github.com/your-org/harxhar.git"], check=True)
os.chdir("harxhar/colab")

# Install dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas", "scikit-learn", "pyarrow", "numba", "tqdm"]
)

# Add colab to path
sys.path.insert(0, ".")

In [ ]:
# Parameters
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
DATA_PATH = "all30min"
PERIODS_PER_DAY = 48
TRAIN_WINDOW = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Horizon: {HORIZON}, Train window: {TRAIN_WINDOW_DAYS} days = {TRAIN_WINDOW} periods")

In [ ]:
# export
"""Ridge regression volatility backtest executor.

Method-specific glue around the shared scaffold in :mod:`src.executor`.
Only the model factory + default hyperparams + ``main()`` wrapper live
here; everything else (CLI parsing, loading, features, backtest loop,
smearing, reduce JSON, segment dispatch) is owned by ``src.executor``.
"""

from __future__ import annotations

import json

import numpy as np
from sklearn.linear_model import Ridge

from src.executor import ExecutorConfig, run_executor
from src.loading import parse_exog_cols
from src.scaling import run_backtest

# Method-specific data-prep config. Lives here (not in src.executor) so the
# ridge spec sits next to the ridge model code. src.executor.CONFIGS imports
# this at runtime to keep a method-name → config registry for drift checks.
CONFIG = ExecutorConfig(
    method="ridge",
    add_calendar=True,
    target_use_diurnal=False,
    target_winsor_window=None,
    dropna_with_exog=True,
    refit_frequency=1,
    calendar_encoding="rich",
)

# Per-method default hyperparams. Tuned overrides from --params-file are
# merged on top via dict.update().
DEFAULT_RIDGE_PARAMS: dict = dict(alpha=1.0)

In [ ]:
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

adj_rv, baseline = robust_transform(df, "RV", is_target=True)
df["adj_RV"] = adj_rv
df["baseline"] = baseline
df[["t", "RV", "adj_RV", "baseline"]].head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from src.transforms import generate_har_features, resolve_har_lags

df, feature_names = generate_har_features(df, target_col="adj_RV")
print(f"HAR lags: {resolve_har_lags()}")
print(f"Feature names: {feature_names}")

# Drop burn-in rows
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)

# Correlation heatmap
corr_cols = ["adj_RV"] + feature_names
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", ax=ax, cmap="coolwarm")
ax.set_title("HAR Feature Correlations")
plt.tight_layout()
plt.show()

In [ ]:
from src.transforms import apply_horizon_shift

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)
print(f"After horizon shift: X {X.shape}, y {y.shape}")
print(f"Date range: {dates.iloc[0]} to {dates.iloc[-1]}")

In [ ]:
from sklearn.linear_model import Ridge

from src.scaling import run_backtest
from src.evaluation import apply_duan_smearing


def model_fn():
    return Ridge(alpha=1.0)


preds = run_backtest(model_fn, X, y, train_win=TRAIN_WINDOW, refit_frequency=1, use_scaling=True)

oos_start = TRAIN_WINDOW
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

# Plot predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(dates_oos[-2000:], y_oos[-2000:], alpha=0.7, label="True (adj)")
axes[0].plot(dates_oos[-2000:], preds[-2000:], alpha=0.7, label="Pred (adj)")
axes[0].set_title("Adjusted-scale predictions (last 2000)")
axes[0].legend()
axes[1].plot(dates_oos[-2000:], true_raw[-2000:], alpha=0.7, label="True (raw)")
axes[1].plot(dates_oos[-2000:], pred_raw[-2000:], alpha=0.7, label="Pred (raw)")
axes[1].set_title("Raw-scale predictions (last 2000)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from src.evaluation import calculate_metrics

results_df = pd.DataFrame(
    {
        "date": dates_oos,
        "horizon": HORIZON,
        "true_adj": y_oos,
        "pred_adj": preds,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
# export
def fit_predict_ridge(
    X_chunk: np.ndarray,
    y_chunk: np.ndarray,
    train_win_periods: int,
    hyperparams: dict,
) -> np.ndarray:
    """Walk-forward backtest with Ridge. Returns OOS predictions.

    Wraps :func:`src.scaling.run_backtest` with the Ridge-specific
    settings (``use_scaling=True`` -- linear models need feature
    standardization; refit frequency from
    ``hyperparams['_refit_frequency']``).

    Notes
    -----
    Ridge's default solver is closed-form, so ``random_state`` is
    irrelevant for reproducibility. We strip any internal control keys
    (``_*``) before forwarding to ``Ridge(**...)``.
    """
    refit_frequency = int(hyperparams.get("_refit_frequency", CONFIG.refit_frequency))
    # Strip our internal control keys before passing to Ridge.
    model_kwargs = {k: v for k, v in hyperparams.items() if not k.startswith("_")}

    def model_fn():
        return Ridge(**model_kwargs)

    return run_backtest(
        model_fn,
        X_chunk,
        y_chunk,
        train_win=train_win_periods,
        refit_frequency=refit_frequency,
        use_scaling=True,
    )

In [ ]:
# Smoke-test fit_predict_ridge on synthetic data.
rng = np.random.default_rng(42)
n, k = 600, 4
X_demo = rng.normal(size=(n, k))
y_demo = rng.normal(size=n)
_hp = dict(DEFAULT_RIDGE_PARAMS, _refit_frequency=10)
preds = fit_predict_ridge(X_demo, y_demo, train_win_periods=500, hyperparams=_hp)
assert preds.shape == (n - 500,), preds.shape
assert np.all(np.isfinite(preds)), "predictions must be finite"
print(f"fit_predict_ridge OK: preds.shape={preds.shape}, mean={preds.mean():.4f}")

In [ ]:
# export
def compute(args) -> None:

    tuned_params: dict = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    # Method defaults <- tuned overrides; refit-frequency from CLI sentinel.
    hyperparams = dict(DEFAULT_RIDGE_PARAMS)
    hyperparams.update(tuned_params)
    refit_frequency = args.refit_frequency if args.refit_frequency is not None else CONFIG.refit_frequency
    hyperparams["_refit_frequency"] = refit_frequency

    exog_cols = parse_exog_cols(args.exog_cols)

    run_executor(
        method_name="ridge",
        fit_predict=fit_predict_ridge,
        hyperparams=hyperparams,
        data_path=args.data_path,
        output_file=args.output_file,
        horizon=args.horizon,
        train_window=args.train_window,
        start=args.start,
        end=args.end,
        exog_cols=exog_cols,
        segment=args.segment,
        lag_scope=args.lag_scope,
        **CONFIG.as_data_prep_kwargs(),
        seed=args.seed,
    )

In [ ]:
import sys

sys.path.insert(0, "..")
from _exporter import export_notebook

export_notebook("ml_ridge.ipynb", "../../src/ml_ridge.py")